# 03_baseline_roster_optimizer.ipynb

Let's create a baseline roster optimizer using linear programming. 

Start with just using data for a single season.

# Expected value of a player
Per the rules of the fantasy hockey league, skaters and goalies have different point systems.

## Skaters

The scoring for skaters (forwards and defencemen) is defined as:
- Goal: 2 Points
- Assist: 1 Point.


The expected value $v$ of a skater $i$ will be defined as:
$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[G_i] + 1 \times \mathbb{E}[A_i], 
$$

with

$$
\mathbb{E}[G_i] = \text{GPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[A_i] = \text{APG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[G]$ is the expected goals, $\mathbb{E}[A]$ is the expected assists,  $\text{GPG}$ is the goals per game in the regular season, $\text{APG}$ is the assists per game in the regular season, and $\mathbb{E}[\text{Games Played}]$ is the expected number of games player $i$'s team will play in the playoffs.

## Goalies
The scoring for goalies is defined as:
- Win: 1 Point
- Assist: 1 Point
- Shutout: 2 Points


The expected value $v$ of a goalie $i$ will be defined as:

$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[\text{SO}_i] + 1 \times \mathbb{E}[W_i] + 1 \times \mathbb{E}[A_i], 
$$

with

$$
\mathbb{E}[\text{SO}_i] = \text{SOPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[W_i] = \text{WPG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[\text{SO}]$ is the expected shutouts, $\mathbb{E}[W]$ is the expected wins,  $\text{SOPG}$ is the shutouts per game in the regular season, and $\text{WPG}$ is the wins per game in the regular season.

# The LP set-up
The roster is selected once at the beginning of the playoffs.

- 15 skaters total
  - 9 forwards
  - 6 defence
- 2 goalies

This presents a straightforward linear programming problem.

## Objective 
We will simplify the above notation and just call $v_i$ the expected value of player $i$.

Let $x_i=1$ if you pick player $i$, otherwise $x_i=0$. 

The objective function is then simply

$$
\text{max}\sum_i v_ix_i.
$$

## Constraints
The number of skaters in each position are our constraints, which can be written out as follows:

$$
\text{Total number of players}: \sum_i x_i = 17,
$$

$$
\text{Total number of forwards}: \sum_{i \in \text{forwards}} x_i = 9,
$$

$$
\text{Total number of defence}: \sum_{i \in \text{defence}} x_i = 6,
$$

$$
\text{Total number of goalies}: \sum_{i \in \text{goalies}} x_i = 2.
$$

We can also add strategic constraints. 

To limit the number of players we select per team as a way to hedge our roster selections (in case of a bracket upset for example):

$$
\text{Limit per team}: \sum_{i \in \text{Team T}} x_i \leq M_{\text{T}} \quad \forall \quad \text{T},
$$

where $M_{\text{T}}$ is our defined maximum number of players for a given team, $\text{T}$.

We can also impose a constraint that since we only choose two goalies total, it is best to not select two goalies who play for this same team.

Similarly, we do not want to pick a goalie who is not the starting goalie for that team.

# The Baseline Plan

- For skaters, compute the goals and assists per game in the regular season. 
- For goalies, compute the wins and shutouts per game in the regular season.

We will make the naive assumption that these rates continue in the playoffs. This assumption is what makes this a baseline model. In future versions, the expected value of a player in the playoffs will be the output of a machine learning model. 

- Multiply the rates by an estimate of how many games the given team's expected number of games played in the playoffs to get the expected value per player.

For a baseline implementation, maybe I could just use a sportsbook's predictions on playoff outcomes to estimate what round each team will make in the playoffs and then make an assumption that each series lasts say 6 games.

Or for running this LP problem on previous seasons I can just use the actual number of games each team played.

With the expected value per player calculated, we can then use scipy to solve the LP problem.


# Load data

Let's try this for the 2024-2025 season


In [359]:
import pandas as pd

from nhl_pool.common import stats_file_path

In [360]:
# SKATERS REGULAR SEASON
params = {
    "season": 20242025,
    "season_type": "regular",
    "player_type": "skaters"
}
skaters_regular_raw = pd.read_csv(stats_file_path(params))

# GOALIES REGULAR SEASON
params = {
    "season": 20242025,
    "season_type": "regular",
    "player_type": "goalies"
}
goalies_regular_raw = pd.read_csv(stats_file_path(params))

# SKATERS PLAYOFFS
params = {
    "season": 20242025,
    "season_type": "playoffs",
    "player_type": "skaters"
}
skaters_playoffs_raw = pd.read_csv(stats_file_path(params))

# GOALIES PLAYOFFS
params = {
    "season": 20242025,
    "season_type": "playoffs",
    "player_type": "goalies"
}
goalies_playoffs_raw = pd.read_csv(stats_file_path(params))

## Pre-processing

### Handling duplicated players
In the regular season files, a player might show up more than one if they were traded. Let's remedy that by collapsing such that a player only appears once.

In [361]:
print(skaters_regular_raw['playerId'].duplicated().sum())
print(goalies_regular_raw['playerId'].duplicated().sum())

107
9


In [362]:
from nhl_pool.processing.players import collapse_players

skaters_regular = collapse_players(skaters_regular_raw, collapse_type="skater")
goalies_regular = collapse_players(goalies_regular_raw, collapse_type="goalie")

# Sanity checks
print(skaters_regular['playerId'].duplicated().sum())
print(goalies_regular['playerId'].duplicated().sum())

0
0


### positionCodes for Forwads

The league playoff points makes no distinction between C, LW, and RW. So we need to map these positions to simply forward, "F"

In [363]:
from nhl_pool.processing.players import map_positions

skaters_regular = map_positions(skaters_regular)
skaters_playoffs = map_positions(skaters_playoffs_raw)

skaters_regular["positionCode"].value_counts()
skaters_playoffs["positionCode"].value_counts()

positionCode
F    216
D    116
Name: count, dtype: int64

### Dropping players who played less than X games.

We want to select players who will regularly play. If a player only played in one game and got a point or two in that game they would have a high expected value because we assume a per game basis.

In [364]:
min_gamesPlayed = 15

skaters_regular = skaters_regular[skaters_regular["gamesPlayed"] >= min_gamesPlayed]
goalies_regular = goalies_regular[goalies_regular["gamesPlayed"] >= min_gamesPlayed]


### Dropping unneeded columns
For skaters we only need goals, assists, gamesPlayed and ID columns

For goalies we only need wins, shutouts, assists, gamesPlayed and ID columns

In [365]:
goalies_regular.columns

Index(['playerId', 'firstName', 'lastName', 'teamAbbrev', 'season',
       'seasonType', 'positionCode', 'gamesPlayed', 'gamesStarted', 'wins',
       'losses', 'overtimeLosses', 'goalsAgainstAverage', 'savePercentage',
       'shotsAgainst', 'saves', 'goalsAgainst', 'shutouts', 'goals', 'assists',
       'points', 'penaltyMinutes', 'timeOnIce', 'sourceFile'],
      dtype='str')

In [366]:
skaters_columns_to_keep = ['playerId', 'firstName', 'lastName', 'teamAbbrev', 'goals', 'assists', 'gamesPlayed']
goalies_columns_to_keep = ['playerId', 'firstName', 'lastName', 'teamAbbrev', 'wins', 'shutouts', 'assists', 'gamesPlayed']


skaters_regular = skaters_regular[skaters_columns_to_keep]
goalies_regular = goalies_regular[goalies_columns_to_keep]

skaters_regular.head()

,playerId,firstName,lastName,teamAbbrev,goals,assists,gamesPlayed
0,8473986,Alex,Killorn,ANA,19,18,82
1,8475462,Radko,Gudas,ANA,1,15,81
2,8476458,Ryan,Strome,ANA,10,31,82
3,8476934,Brock,McGinn,ANA,4,4,26
4,8477527,Ross,Johnston,ANA,1,3,43


# Expected Value of Players
## Skaters
For skaters, we only need to compute the goals and assists per game

In [367]:
skaters_regular["goalsPerGame"] = skaters_regular["goals"] / skaters_regular["gamesPlayed"]
skaters_regular["assistsPerGame"] = skaters_regular["assists"] / skaters_regular["gamesPlayed"]
skaters_regular.head()

,playerId,firstName,lastName,teamAbbrev,goals,assists,gamesPlayed,goalsPerGame,assistsPerGame
0,8473986,Alex,Killorn,ANA,19,18,82,0.231707,0.219512
1,8475462,Radko,Gudas,ANA,1,15,81,0.012346,0.185185
2,8476458,Ryan,Strome,ANA,10,31,82,0.121951,0.378049
3,8476934,Brock,McGinn,ANA,4,4,26,0.153846,0.153846
4,8477527,Ross,Johnston,ANA,1,3,43,0.023256,0.069767


Now we can create the expected value of each skater per game, valuePerGame

In [368]:
skaters_regular["valuePerGame"] = 2*skaters_regular["goalsPerGame"] + skaters_regular["assistsPerGame"]

skaters_regular.sort_values(by="valuePerGame", ascending=False).head()

,playerId,firstName,lastName,teamAbbrev,goals,assists,gamesPlayed,goalsPerGame,assistsPerGame,valuePerGame
276,8477934,Leon,Draisaitl,EDM,52,54,71,0.732394,0.760563,2.225352
640,8476453,Nikita,Kucherov,TBL,37,84,78,0.474359,1.076923,2.025641
353,8478864,Kirill,Kaprizov,MIN,25,31,41,0.609756,0.756098,1.975610
278,8478402,Connor,McDavid,EDM,26,74,67,0.388060,1.104478,1.880597
192,8477492,Nathan,MacKinnon,COL,32,84,79,0.405063,1.063291,1.873418


## Goalies
For goalies, we only need to compute the wins, shutouts, and assists per game

In [369]:
goalies_regular["winsPerGame"] = goalies_regular["wins"] / goalies_regular["gamesPlayed"]
goalies_regular["shutoutsPerGame"] = goalies_regular["shutouts"] / goalies_regular["gamesPlayed"]
goalies_regular["assistsPerGame"] = goalies_regular["assists"] / goalies_regular["gamesPlayed"]
goalies_regular.head()

,playerId,firstName,lastName,teamAbbrev,wins,shutouts,assists,gamesPlayed,winsPerGame,shutoutsPerGame,assistsPerGame
0,8476434,John,Gibson,ANA,11,0,1,29,0.379310,0.000000,0.034483
1,8480843,Lukas,Dostal,ANA,23,1,0,54,0.425926,0.018519,0.000000
2,8476914,Joonas,Korpisalo,BOS,11,3,0,27,0.407407,0.111111,0.000000
3,8480280,Jeremy,Swayman,BOS,22,4,0,58,0.379310,0.068966,0.000000
4,8480045,Ukko-Pekka,Luukkonen,BUF,24,2,1,55,0.436364,0.036364,0.018182


Now we can create the expected value of each goalie per game, valuePerGame

In [370]:
goalies_regular["valuePerGame"] = 2*goalies_regular["shutoutsPerGame"] + goalies_regular["winsPerGame"] + goalies_regular["assistsPerGame"]

goalies_regular.sort_values(by="valuePerGame", ascending=False).head()

,playerId,firstName,lastName,teamAbbrev,wins,shutouts,assists,gamesPlayed,winsPerGame,shutoutsPerGame,assistsPerGame,valuePerGame
88,8476945,Connor,Hellebuyck,WPG,47,8,1,63,0.746032,0.126984,0.015873,1.015873
74,8476932,Anthony,Stolarz,TOR,21,4,1,34,0.617647,0.117647,0.029412,0.882353
29,8475311,Darcy,Kuemper,LAK,31,5,1,50,0.620000,0.100000,0.020000,0.840000
91,8480313,Logan,Thompson,WSH,31,2,1,43,0.720930,0.046512,0.023256,0.837209
70,8476883,Andrei,Vasilevskiy,TBL,38,6,2,63,0.603175,0.095238,0.031746,0.825397


# Expected number of games played

For now, let's hardcode the 2024-2025 actual brackets based on how many series each team played.

We'll make an assumption of and average of 6 games per series.

In the full system, this will get replaced by the output of the playoff bracket simulation results.





In [371]:
expected_playoff_series = {
    "WPG": 2,
    "STL": 1,
    "DAL": 3,
    "COL": 1,
    "VGK": 2,
    "MIN": 1,
    "LAK": 1,
    "EDM": 4,
    "TOR": 2,
    "OTT": 1,
    "TBL": 1,
    "FLA": 4,
    "WSH": 2,
    "MTL": 1,
    "CAR": 3,
    "NJD": 1
}

games_per_series = 6

# Roster Optimization

Now we can move on to actually selecting our roster.

First things first, we need to create clean dataFrames of just the skaters and goalies who will play in the playoffs.

In the full system, the API calls method will have to be extended to get the roster of the team. For now, let's just use the playoffs tables and get the player information/team information from there.

In [372]:

playoff_cols = ["playerId", "firstName", "lastName", "teamAbbrev", "positionCode"]
skaters_playoffs = skaters_playoffs[playoff_cols]
goalies_playoffs = goalies_playoffs_raw[playoff_cols]

skaters_playoffs.head()

,playerId,firstName,lastName,teamAbbrev,positionCode
0,8470613,Brent,Burns,CAR,D
1,8473533,Jordan,Staal,CAR,F
2,8475200,Dmitry,Orlov,CAR,D
3,8475791,Taylor,Hall,CAR,F
4,8476873,Mark,Jankowski,CAR,F


### Merging valuePerGame

Now we can left join the playoffs lists to the regular season to get the valuePerGame column for just the playoffs players.

In [373]:
skaters_playoffs = skaters_playoffs.merge(skaters_regular[["playerId", "valuePerGame"]],
                                          on="playerId",
                                          how="left")
skaters_playoffs["valuePerGame"] = skaters_playoffs["valuePerGame"].fillna(0)
skaters_playoffs.sort_values(by='valuePerGame', ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
76,8477934,Leon,Draisaitl,EDM,F,2.225352
232,8476453,Nikita,Kucherov,TBL,F,2.025641
142,8478864,Kirill,Kaprizov,MIN,F,1.975610
80,8478402,Connor,McDavid,EDM,F,1.880597
28,8477492,Nathan,MacKinnon,COL,F,1.873418
...,...,...,...,...,...,...
45,8475755,Alexander,Petrovic,DAL,D,0.000000
305,8479994,Jaret,Anderson-Dolan,WPG,F,0.000000
297,8476952,Dominic,Toninato,WPG,F,0.000000
327,8480823,Alexander,Alexeyev,WSH,D,0.000000


In [374]:
skaters_playoffs["teamAbbrev"].unique()

<StringArray>
['CAR', 'COL', 'DAL', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NJD', 'OTT', 'STL',
 'TBL', 'TOR', 'VGK', 'WPG', 'WSH']
Length: 16, dtype: str

In [375]:
goalies_playoffs = goalies_playoffs.merge(goalies_regular[["playerId", "valuePerGame"]],
                                          on="playerId",
                                          how="left")
goalies_playoffs["valuePerGame"] = goalies_playoffs["valuePerGame"].fillna(0)
goalies_playoffs.sort_values(by='valuePerGame', ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
23,8476945,Connor,Hellebuyck,WPG,G,1.015873
19,8476932,Anthony,Stolarz,TOR,G,0.882353
9,8475311,Darcy,Kuemper,LAK,G,0.840000
26,8480313,Logan,Thompson,WSH,G,0.837209
17,8476883,Andrei,Vasilevskiy,TBL,G,0.825397
8,8475683,Sergei,Bobrovsky,FLA,G,0.814815
21,8478499,Adin,Hill,VGK,G,0.800000
15,8476999,Linus,Ullmark,OTT,G,0.795455
2,8475809,Scott,Wedgewood,COL,G,0.750000
0,8475883,Frederik,Andersen,CAR,G,0.727273


### Joining tables together
Now we want to put skaters and goalies into one table

In [376]:
players = pd.concat([skaters_playoffs, goalies_playoffs], ignore_index=True)
players

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
0,8470613,Brent,Burns,CAR,D,0.426829
1,8473533,Jordan,Staal,CAR,F,0.653333
2,8475200,Dmitry,Orlov,CAR,D,0.447368
3,8475791,Taylor,Hall,CAR,F,0.779221
4,8476873,Mark,Jankowski,CAR,F,0.483333
...,...,...,...,...,...,...
354,8481033,Akira,Schmid,VGK,G,0.000000
355,8476945,Connor,Hellebuyck,WPG,G,1.015873
356,8477480,Eric,Comrie,WPG,G,0.650000
357,8479292,Charlie,Lindgren,WSH,G,0.564103


### Mapping expected games played by team and getting the total expected value of each player

In [377]:
players["expectedGamesPlayed"] = players["teamAbbrev"].map(expected_playoff_series) * games_per_series
players

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed
0,8470613,Brent,Burns,CAR,D,0.426829,18
1,8473533,Jordan,Staal,CAR,F,0.653333,18
2,8475200,Dmitry,Orlov,CAR,D,0.447368,18
3,8475791,Taylor,Hall,CAR,F,0.779221,18
4,8476873,Mark,Jankowski,CAR,F,0.483333,18
...,...,...,...,...,...,...,...
354,8481033,Akira,Schmid,VGK,G,0.000000,12
355,8476945,Connor,Hellebuyck,WPG,G,1.015873,12
356,8477480,Eric,Comrie,WPG,G,0.650000,12
357,8479292,Charlie,Lindgren,WSH,G,0.564103,12


In [378]:
players["expectedValue"] = players["valuePerGame"] * players["expectedGamesPlayed"]

players.sort_values(by="expectedValue", ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
76,8477934,Leon,Draisaitl,EDM,F,2.225352,24,53.408451
80,8478402,Connor,McDavid,EDM,F,1.880597,24,45.134328
100,8479314,Matthew,Tkachuk,FLA,F,1.519231,24,36.461538
94,8477933,Sam,Reinhart,FLA,F,1.518987,24,36.455696
90,8477493,Aleksander,Barkov,FLA,F,1.358209,24,32.597015
...,...,...,...,...,...,...,...,...
65,8475169,Evander,Kane,EDM,F,0.000000,24,0.000000
45,8475755,Alexander,Petrovic,DAL,D,0.000000,18,0.000000
25,8476455,Gabriel,Landeskog,COL,F,0.000000,6,0.000000
331,8484186,Ryan,Leonard,WSH,F,0.000000,12,0.000000


# Optimization via Linear Programming (Baseline)

We can solve this constrained optimization problem using Scipy.

Let's extract the columns we need

In [379]:
ev = players["expectedValue"].to_numpy(dtype=float)
pos = players["positionCode"]
team = players["teamAbbrev"]
N = len(players)

### Objective function
Maximising the expected value times the decision variable is the objective function.

Scipy's ```milp``` is a minimiser, so simply convert this to a minimisation problem.

In [380]:
c = -ev

### Indicator vectors for each position type
This is how we are going to force the position contraints

In [381]:
import numpy as np
is_forward = np.array(pos == "F").astype(int)
is_defence = np.array(pos == "D").astype(int)
is_goalie = np.array(pos == "G").astype(int)

## Build constraints as a matrix
Think about the constraints as being implemented as 
$$
Ax = b
$$

where $x$ is the decision variable (1 if select player else 0), $A$ are all the indicator vectors, and $b$ are the optimization constraints.

In [382]:
A_eq = np.vstack([np.ones_like(is_forward), is_forward, is_defence, is_goalie])

b_eq = np.array([17, 9, 6, 2])

print(A_eq.shape)
print(b_eq.shape)

(4, 359)
(4,)


## Solving with milp

In [383]:
from scipy.optimize import milp, Bounds, LinearConstraint

bounds = Bounds(lb=np.zeros(N), ub=np.ones(N))
integrality = np.ones(N, dtype=int)

# Equality is enforced with lb=ub
constraints_roster = LinearConstraint(A_eq, lb=b_eq, ub=b_eq) 

result = milp(c=c, constraints=constraints_roster, integrality=integrality, bounds=bounds)

In [384]:
roster_mask = result.x == 1

### Investigate the "optimal" roster

In [385]:
roster = players[roster_mask].sort_values(by='teamAbbrev')
print(f"Total Expected Value = {roster["expectedValue"].sum():.4f}")
roster

Total Expected Value = 446.9164


,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
46,8475794,Tyler,Seguin,DAL,F,1.500000,18,27.000000
51,8478420,Mikko,Rantanen,DAL,F,1.463415,18,26.341463
55,8480027,Jason,Robertson,DAL,F,1.402439,18,25.243902
59,8481581,Thomas,Harley,DAL,D,0.846154,18,15.230769
338,8475717,Calvin,Pickard,EDM,G,0.638889,24,15.333333
80,8478402,Connor,McDavid,EDM,F,1.880597,24,45.134328
83,8480803,Evan,Bouchard,EDM,D,0.987805,24,23.707317
76,8477934,Leon,Draisaitl,EDM,F,2.225352,24,53.408451
66,8475218,Mattias,Ekholm,EDM,D,0.646154,24,15.507692
78,8478013,Jake,Walman,EDM,D,0.723077,24,17.353846


# Optimization via Linear Programming (with additional constraints)

As a way to hedge our bets, and to intelligently optimize the roster we will introduce some extra constraints.

1. Only selecting a maximum number of players from each team
2. Only select one goalie per team
3. Only select starting goaltenders (to be implemented later, need to label starting goalies manually).

We'll start with maximum number of players for each team:

In [398]:
teams = sorted(players["teamAbbrev"].unique())
team = players["teamAbbrev"].to_numpy()

max_per_team = 6

A_team = np.vstack([(team == t).astype(int) for t in teams])
con_team_max = LinearConstraint(A_team, lb=-np.inf, ub=max_per_team)

Now only one goalie per team:

In [399]:
A_goalie_team = np.vstack([((team == t) & (pos == "G")).astype(int) for t in teams])
con_goalie_max = LinearConstraint(A_goalie_team, lb=-np.inf, ub=1)

## Optimize roster

Add it with our original constraints

In [400]:
result = milp(
    c=c,
    constraints=[constraints_roster, con_team_max, con_goalie_max],
    integrality=integrality,
    bounds=bounds
)

roster_mask = result.x == 1

roster = players[roster_mask].sort_values(by='teamAbbrev')
print(f"Total Expected Value = {roster["expectedValue"].sum():.4f}")
roster

Total Expected Value = 446.4877


,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
44,8475168,Matt,Duchene,DAL,F,1.365854,18,24.585366
46,8475794,Tyler,Seguin,DAL,F,1.500000,18,27.000000
51,8478420,Mikko,Rantanen,DAL,F,1.463415,18,26.341463
55,8480027,Jason,Robertson,DAL,F,1.402439,18,25.243902
59,8481581,Thomas,Harley,DAL,D,0.846154,18,15.230769
83,8480803,Evan,Bouchard,EDM,D,0.987805,24,23.707317
338,8475717,Calvin,Pickard,EDM,G,0.638889,24,15.333333
80,8478402,Connor,McDavid,EDM,F,1.880597,24,45.134328
76,8477934,Leon,Draisaitl,EDM,F,2.225352,24,53.408451
66,8475218,Mattias,Ekholm,EDM,D,0.646154,24,15.507692


# Takeaway Notes and Future Work

## Expected number of games played

As we can see, the expected number of games played per team is an extremely important factor in deciding the optimal player.
In this baseline example we had the benefit of knowing exactly how many series a team played in the playoffs. When using this workflow for an actual prediction, we will only have a list of teams. Our playoff bracket simulation framework will have to be good for this entire system to work.


## The "True" optimal roster
It would be interesting to see given the true playoff stats, what was the optimal roster? Could compare and contrast to the prediction at the start of the playoffs.
